# Multi-Fidelity Hyperparameter Optimisation: Hyperband, ASHA, and BOED (C04 Topic 12)

**Citations:** Jamieson & Talwalkar (2016 Successive Halving); Li et al. (2017 Hyperband); Bergstra et al. (2011 TPE); Li et al. (2020 ASHA).

### Purpose & Topic Bridges
Training modern deep neural networks across hundreds of hyperparameter configurations is computationally prohibitive if every trial runs to full convergence. **Multi-Fidelity Hyperparameter Optimization (HPO)** leverages cheap low-fidelity proxies (e.g. early epochs, dataset sub-samples) to rapidly eliminate unpromising candidates.

**Successive Halving (SHA)** iteratively prunes the worst-performing configurations by factor $\eta$, allocating geometrically increasing budgets to survivors. **Hyperband** resolves the exploration-exploitation dilemma of SHA by hedging across multiple allocation brackets $(n_s, r_s)$, preventing the premature elimination of slow-starting models. **ASHA (Asynchronous Successive Halving)** eliminates straggler synchronization bottlenecks in distributed clusters via non-blocking promotions. **TPE (Tree-structured Parzen Estimator)** models non-parametric density ratios $\ell(x) / g(x)$ for fast sample-efficient Bayesian optimization.

**Decision Boundary:** Use **ASHA** for large distributed GPU clusters to eliminate idle synchronization. Use **Hyperband** when learning curve dynamics vary widely across configurations (e.g. warmup vs aggressive LR). Use **TPE/BOHB** when configuration evaluation is expensive and sample efficiency is paramount. When NOT to use: Never use single-bracket **Successive Halving** without bracket hedging when comparing fast-saturating models against slow-starting cosine schedules; SHA prematurely kills the best final models at epoch 1.

## 2. Mathematical Foundations & Governing Equations

1. **Successive Halving Elimination:** At rung $k$, given $n_k$ models and budget $r_k = r_{\min} \eta^k$, retain top $k_{\text{surv}} = \max(1, \lfloor n_k / \eta \rfloor)$ models.
2. **Hyperband Bracket Allocation:** For total resource $R$, $s_{\max} = \lfloor \log_\eta(R) \rfloor$, budget $B = (s_{\max} + 1) R$:
$$n_s = \left\lceil \frac{B}{R} \frac{\eta^s}{s + 1} \right\rceil, \quad r_s = R \eta^{-s} \quad \text{for } s \in [s_{\max}, \dots, 0]$$
3. **TPE Acquisition via Density Ratio:** With split threshold $y^*$:
$$\text{EI}(x) \propto \frac{\ell(x)}{g(x)} = \frac{p(x \mid y < y^*)}{p(x \mid y \ge y^*)} \implies \alpha(x) = \log \ell(x) - \log g(x)$$
4. **ASHA Promotion Criterion:** A job completing rung $k$ is promoted if $\text{rank}(\mathcal{L}_i) \le \lfloor |\text{completed}_k| / \eta \rfloor$.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np, torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/02_constrained_pareto_and_blackbox_optimisation/05_multi_fidelity_hyperparameter_optimisation_hyperband_asha_and_boed.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
def compute_sha_top_k(losses, eta=3):
    k = max(1, losses.shape[0] // eta)
    sorted_losses, sorted_indices = torch.sort(losses)
    return sorted_indices[:k], sorted_losses[:k]

def compute_hyperband_schedule(R=81, eta=3):
    s_max = int(math.floor(math.log(R, eta)))
    B = (s_max + 1) * R
    schedule = []
    for s in range(s_max, -1, -1):
        n_s = int(math.ceil((B / R) * ((eta ** s) / (s + 1))))
        r_s = int(R * (eta ** (-s)))
        schedule.append([float(n_s), float(r_s)])
    return torch.tensor(schedule, dtype=torch.float32)

def compute_tpe_log_ratio(x_query, good_samples, bad_samples, bandwidth=0.5):
    def log_kde(x, samples):
        diff = (x.unsqueeze(-1) - samples.unsqueeze(0)) / bandwidth
        log_kernel = -0.5 * diff ** 2 - 0.5 * math.log(2.0 * math.pi) - math.log(bandwidth)
        return torch.logsumexp(log_kernel, dim=-1) - math.log(samples.shape[0])
    return log_kde(x_query, good_samples) - log_kde(x_query, bad_samples)

def check_asha_promotion(rung_losses, candidate_loss, eta=3):
    k = max(1, rung_losses.shape[0] // eta)
    sorted_losses, _ = torch.sort(rung_losses)
    return 1.0 if candidate_loss <= sorted_losses[k - 1].item() else 0.0

class LinearAnchor(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_f, in_f))
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        return torch.matmul(x, self.weight.T) + self.bias

print('MultiFidelityUnifiedModule routines defined successfully.')

In [ ]:
# 1. SHA
sha_idx, sha_losses = compute_sha_top_k(weights['sha_losses'], eta=3)
err_sha = max(float(torch.max(torch.abs(sha_idx.to(torch.float32) - weights['golden_sha_idx']))), float(torch.max(torch.abs(sha_losses - weights['golden_sha_losses']))))

# 2. Hyperband Schedule
hb_sched = compute_hyperband_schedule(R=81, eta=3)
err_hb = float(torch.max(torch.abs(hb_sched - weights['golden_hb_schedule'])))

# 3. TPE Log Ratio
tpe_ratio = compute_tpe_log_ratio(weights['x_query'], weights['good_samples'], weights['bad_samples'], bandwidth=0.5)
err_tpe = float(torch.max(torch.abs(tpe_ratio - weights['golden_tpe_log_ratio'])))

# 4. ASHA Promotion
p0 = check_asha_promotion(weights['rung_losses'], float(weights['candidate_losses'][0]), eta=3)
p1 = check_asha_promotion(weights['rung_losses'], float(weights['candidate_losses'][1]), eta=3)
err_asha = float(torch.max(torch.abs(torch.tensor([p0, p1]) - weights['golden_asha_promoted'])))

# 5. Linear Anchor
anchor = LinearAnchor(8, 4)
anchor.weight.data.copy_(weights['anchor_w'].T)
anchor.bias.data.copy_(weights['anchor_b'])
anchor_out = anchor(weights['anchor_x']).detach()
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])))

for name, err in [('SHA', err_sha), ('Hyperband', err_hb), ('TPE', err_tpe), ('ASHA', err_asha), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 5 invariants verified! SHA: {err_sha:.2e}, HB: {err_hb:.2e}, TPE: {err_tpe:.2e}, ASHA: {err_asha:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'SHA Top-K Pruning': lambda: compute_sha_top_k(weights['sha_losses']),
    'Hyperband Schedule': lambda: compute_hyperband_schedule(81, 3),
    'TPE Density Ratio': lambda: compute_tpe_log_ratio(weights['x_query'], weights['good_samples'], weights['bad_samples']),
    'ASHA Promotion Check': lambda: check_asha_promotion(weights['rung_losses'], 0.12),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Slow-Starter Death Trap in Single-Rung Successive Halving vs Hyperband
# Model A (aggressive LR): drops fast at epoch 1 (0.23), but plateaus poorly at epoch 81 (0.186).
# Model B (cosine warmup): starts slow at epoch 1 (0.602), but reaches state-of-the-art at epoch 81 (0.045).
def loss_curve_A(ep): return 0.18 + 0.05 / (ep ** 0.5)
def loss_curve_B(ep): return 0.02 + 0.60 * (1.0 / (1.0 + (ep / 10.0) ** 1.5))

# Single-Rung SHA (r_min = 1 epoch, eta = 3): evaluates both at epoch 1, keeps only the top 1/3 (best 1 of 3):
loss_ep1_A, loss_ep1_B = loss_curve_A(1), loss_curve_B(1)
sha_survivor = 'Model A' if loss_ep1_A < loss_ep1_B else 'Model B'
sha_final_loss = loss_curve_A(81) if sha_survivor == 'Model A' else loss_curve_B(81)

# Hyperband Bracket s = 1 (evaluates at r_init = 27 epochs):
loss_ep27_A, loss_ep27_B = loss_curve_A(27), loss_curve_B(27)
hb_survivor = 'Model B' if loss_ep27_B < loss_ep27_A else 'Model A'
hb_final_loss = loss_curve_B(81) if hb_survivor == 'Model B' else loss_curve_A(81)

print(f'[FAILURE DEMONSTRATED] Single-Rung SHA eliminated slow-starting Model B at epoch 1 (L_A={loss_ep1_A:.2f} < L_B={loss_ep1_B:.2f}). Converged to inferior loss: {sha_final_loss:.3f}!')
print(f'[CORRECT] Hyperband bracket s=1 evaluated at epoch 27 (L_B={loss_ep27_B:.2f} < L_A={loss_ep27_A:.2f}). Preserved best model to victory: {hb_final_loss:.3f} (4.1x lower error!)')
assert sha_final_loss > 0.15 and hb_final_loss < 0.05, 'Hyperband hedging failed to outperform naive SHA.'
print('[REMEDY] Always use Hyperband bracket hedging or ASHA with multi-rung entry to protect slow-starting models.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Multi-Fidelity Eliminates Waste:** Successive Halving and Hyperband achieve $10\times\text{--}30\times$ compute speedups by terminating unpromising configurations early.
2. **Bracket Hedging Protects High-Capacity Models:** Hedging across initial resource budgets protects slow-starting learning dynamics from premature elimination.
3. **TPE Reverses Conditioning:** Modeling $p(x \mid y)$ rather than $p(y \mid x)$ allows non-parametric KDEs to compute Expected Improvement without cubic GP matrix inversion.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 04_combinatorial_and_discrete_simulated_annealing_tempering_and_sinkhorn](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/02_constrained_pareto_and_blackbox_optimisation/04_combinatorial_and_discrete_simulated_annealing_tempering_and_sinkhorn.ipynb) | [Next: 06_sharpness_aware_swad_and_weight_averaging_dynamics](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/02_constrained_pareto_and_blackbox_optimisation/06_sharpness_aware_swad_and_weight_averaging_dynamics.ipynb)